## **Comparación de modelos**

En este notebook se realiza un análisis y comparación de 3 modelos posibles para el modelo final.

1. Random Forest
2. XGBoost
3. LightGBM

Usando como métricas de comparación ROC-AUC y PR-AUC.

In [2]:
import pandas as pd

X_train = pd.read_parquet('data/X_train.parquet')
y_train = pd.read_parquet('data/y_train.parquet').squeeze()  # Convertir a Series
X_val = pd.read_parquet('data/X_Val.parquet')
y_val = pd.read_parquet('data/y_Val.parquet').squeeze()  # Convertir a Series


In [3]:
X_train.drop(columns=['TransactionID', 'TransactionDT'], inplace=True)
X_val.drop(columns=['TransactionID', 'TransactionDT'], inplace=True)

print(X_train.shape)
print(y_train.shape)
print(X_val.shape)
print(y_val.shape)

(472432, 403)
(472432,)
(118108, 403)
(118108,)


In [4]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale = neg / pos
print(f"Negativos: {neg}, Positivos: {pos}, scale_pos_weight: {scale:.2f}")

Negativos: 455833, Positivos: 16599, scale_pos_weight: 27.46


In [5]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import lightgbm as lgb

models = {
    'Random Forest': RandomForestClassifier(n_estimators=100,class_weight='balanced', random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(scale_pos_weight=scale, random_state=42, n_jobs=-1,eval_metric='auc'),
    'LightGBM': lgb.LGBMClassifier(scale_pos_weight=scale, random_state=42, n_jobs=-1, verbose=-1)
}



## **Manejo del desbalance: `class_weight` vs `scale_pos_weight`**

Random Forest, XGBoost y LightGBM manejan el desbalance de clases de forma distinta, por eso en la celda de modelos no se usó el mismo parámetro para los tres.

`scale_pos_weight` es un parámetro propio de XGBoost y LightGBM, que penaliza más los errores sobre la clase minoritaria multiplicando su peso por el ratio entre negativos y positivos. Scikit-learn no implementa este parámetro en `RandomForestClassifier`, por lo que ahí se usa `class_weight='balanced'`, que cumple el mismo objetivo pero calculando internamente un peso por clase según su frecuencia en el dataset.

En conclusión, ambos mecanismos buscan lo mismo, compensar que el modelo no ignore la clase minoritaria por ser tan pequeña, pero cada librería lo implementa con su propio parámetro.

In [6]:
from sklearn.metrics import roc_auc_score, average_precision_score
import time

results = []

for name, model in models.items():
    print(f"Entrenando {name}")
    start_time = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start_time
    print(f"Tiempo de entrenamiento: {elapsed:.2f} segundos")

    y_prob = model.predict_proba(X_val)[:, 1]

    auc = roc_auc_score(y_val, y_prob)
    pr_auc = average_precision_score(y_val, y_prob)

    results.append({'Modelo': name, 'ROC AUC': round(auc, 4), 'PR AUC': round(pr_auc, 4)})
    print(f" AUC: {auc:.4f} | PR AUC: {pr_auc:.4f}")

results_df = pd.DataFrame(results)
print("\nResultados comparativos:")
print(results_df)

Entrenando Random Forest
Tiempo de entrenamiento: 122.62 segundos
 AUC: 0.8957 | PR AUC: 0.5031
Entrenando XGBoost
Tiempo de entrenamiento: 34.11 segundos
 AUC: 0.8867 | PR AUC: 0.4896
Entrenando LightGBM
Tiempo de entrenamiento: 23.10 segundos
 AUC: 0.8959 | PR AUC: 0.4891

Resultados comparativos:
          Modelo  ROC AUC  PR AUC
0  Random Forest   0.8957  0.5031
1        XGBoost   0.8867  0.4896
2       LightGBM   0.8959  0.4891


## **Selección del modelo ganador**

Entre la comparación de los 3 modelos, ganó LightGBM con 0.8969 en ROC-AUC, 
lo que indica una mejor capacidad para diferenciar entre clases. Por detrás 
queda Random Forest, que tuvo la puntuación más alta en PR-AUC (0.5031 vs 0.4891).

No se escogió Random Forest porque en un ambiente real, lo más importante para 
un banco es detectar cuáles transacciones son fraudes verdaderos, ya que son los 
casos que más problemas legales y económicos pueden traer. Para ello, LightGBM 
es más práctico: es 6 veces más rápido y lidera en ROC-AUC.

ROC-AUC mide qué tan bien el modelo diferencia entre clases en general, aunque 
puede inflarse por el desbalance de clases. PR-AUC muestra qué tan confiables 
son las predicciones de fraude y cuántos fraudes reales captura el modelo. 
LightGBM está casi a la par con Random Forest en PR-AUC, lo que confirma que 
no sacrifica detección real de fraudes a cambio de velocidad.

En conclusión, LightGBM ofrece las mejores métricas generales con el menor 
costo computacional. La diferencia en PR-AUC con Random Forest (0.016) no 
justifica un modelo 6 veces más lento. El siguiente notebook (`04_model_final.ipynb`) 
profundiza en LightGBM con tuning de hiperparámetros y análisis SHAP.